# 04 — ML Models (LightGBM / XGBoost)
Train gradient boosting models on the full feature matrix. Compare against baselines from notebook 03.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from src.features import build_feature_matrix
from src.backtest import walk_forward_backtest
from src.evaluate import compile_metrics, save_metrics, plot_predictions, plot_shap_summary
from src.models.ml_models import LightGBMForecaster, XGBoostForecaster

df_raw = pd.read_csv('../data/raw/macro_raw.csv', index_col='date', parse_dates=True)
print('Loaded raw data')

In [ ]:
target = 'cpi'
results = []

for horizon in [1, 2, 4]:
    X, y = build_feature_matrix(df_raw, target_col=target, forecast_horizon=horizon)
    print(f'\nHorizon h={horizon}: {X.shape[1]} features, {len(y)} samples')

    for ModelClass, name in [
        (LightGBMForecaster, 'LightGBM'),
        (XGBoostForecaster, 'XGBoost'),
    ]:
        model = ModelClass()
        print(f'  Running {name}...')
        result = walk_forward_backtest(X, y, model, model_name=name, horizon=horizon, min_train_size=60)
        results.append(result)
        print(f'    RMSE={result.rmse:.4f}  MAE={result.mae:.4f}  MAPE={result.mape:.2f}%')

In [ ]:
# Compare all results
metrics_df = compile_metrics(results)
print(metrics_df.to_string(index=False))

In [ ]:
# Plot best ML model predictions
best = min(results, key=lambda r: r.rmse)
plot_predictions(best, title=f'{best.model_name} — {target.upper()} h={best.horizon}')

## SHAP Feature Importance
Train on full dataset (not walk-forward) to get stable SHAP values for the final report/README.

In [ ]:
X_full, y_full = build_feature_matrix(df_raw, target_col='cpi', forecast_horizon=1)
lgbm = LightGBMForecaster()
lgbm.fit(X_full, y_full)
plot_shap_summary(lgbm._model, X_full, model_name='LightGBM_cpi_h1')
print('SHAP plots saved to outputs/figures/')

In [ ]:
# Load baseline metrics and append ML results
try:
    baseline_df = pd.read_csv('../outputs/metrics.csv')
    all_metrics = pd.concat([baseline_df, metrics_df], ignore_index=True)
except FileNotFoundError:
    all_metrics = metrics_df

all_metrics = all_metrics.sort_values(['horizon', 'rmse'])
save_metrics(all_metrics)
all_metrics